In [5]:
print('Hello World')

Hello World


In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

Mounted at /content/gdrive


In [6]:
pip install pillow

In [20]:
import json
import os
import csv
from PIL import Image

# ==========================================
# 1. 設定部分
# ==========================================

# ベースURL (筑波大学サーバーのルート)
BASE_URL = "https://www.u.tsukuba.ac.jp/~s2211717"

# ★変更: すべて同じフォルダ（カレントディレクトリ）で処理します
IMAGE_DIR = "."       # 画像がある場所（. は「ここ」という意味）
OUTPUT_DIR = "."      # 出力する場所
INPUT_CSV = "Anonymous-data.csv"
MANIFEST_FILENAME = "Anonymous-manifest.json"

# CSVの中でCanvasごとのメタデータとして使うキー
CANVAS_METADATA_KEYS = [
    "Title",
    "Creator",
    "Date",
    "Description",
    "Hierarchy_Level",
    "Includes"
]

# ==========================================
# 2. マニフェスト生成ロジック (IIIF v2準拠)
# ==========================================

def create_single_manifest_v2(rows):
    """
    CSVの全行(rows)を受け取り、1つのManifestを作成する
    """
    if not rows:
        return None

    # 1行目からManifest全体の情報を取得
    first_row = rows[0]

    # マニフェスト全体のラベルと説明
    manifest_label = first_row.get("Manifest_Label", "Anonymous Church Archive")
    manifest_description = first_row.get("Manifest_Summary", "")

    # ライセンスと帰属
    license_url = first_row.get("license", "").strip()
    attribution_text = first_row.get("Attribution", "").strip()

    # ページ送り方向
    viewing_direction = first_row.get("viewingDirection", "left-to-right").strip()

    # URL定義 (すべてBASE_URL直下)
    manifest_id = f"{BASE_URL}/{MANIFEST_FILENAME}"
    sequence_id = f"{BASE_URL}/sequence/s1"

    # -------------------------------------------------
    # Canvas（ページ）のリストを作成
    # -------------------------------------------------
    canvases = []

    for i, row in enumerate(rows):
        # ラベル処理
        canvas_label = row.get("label", "").strip()
        if not canvas_label:
            canvas_label = f"p.{i+1}"

        image_filename = row.get("image_filename", "").strip()

        # 画像ファイル名がない行はスキップ
        if not image_filename:
            continue

        # 画像サイズの取得 (スクリプトと同じ場所を探す)
        # . (カレントディレクトリ) とファイル名を結合
        image_path = os.path.join(IMAGE_DIR, image_filename)

        width = 0
        height = 0

        if os.path.exists(image_path):
            try:
                with Image.open(image_path) as img:
                    width, height = img.size
            except Exception as e:
                # 自身のファイルなどを読み込もうとした場合のエラー回避
                print(f"警告: {image_filename} は画像として開けませんでした ({e}) -> スキップ")
                continue
        else:
            print(f"警告: 画像が見つかりません -> {image_filename} (スキップします)")
            continue

        # URL定義 (URLも直下)
        canvas_slug = os.path.splitext(image_filename)[0]
        canvas_id = f"{BASE_URL}/canvas/{canvas_slug}"
        image_url = f"{BASE_URL}/{image_filename}"

        # Canvasごとのメタデータ
        canvas_metadata = []
        for key in CANVAS_METADATA_KEYS:
            if key in row and row[key].strip():
                canvas_metadata.append({
                    "label": key,
                    "value": row[key]
                })

        # Canvasオブジェクト (v2)
        canvas = {
            "@id": canvas_id,
            "@type": "sc:Canvas",
            "label": canvas_label,
            "height": height,
            "width": width,
            "metadata": canvas_metadata,
            "images": [
                {
                    "@type": "oa:Annotation",
                    "motivation": "sc:painting",
                    "resource": {
                        "@id": image_url,
                        "@type": "dctypes:Image",
                        "format": "image/jpeg",
                        "height": height,
                        "width": width
                    },
                    "on": canvas_id
                }
            ]
        }
        canvases.append(canvas)

    if not canvases:
        print("エラー: 有効な画像が1つも見つかりませんでした。")
        return None

    # Manifest全体の構造 (v2)
    manifest = {
        "@context": "http://iiif.io/api/presentation/2/context.json",
        "@id": manifest_id,
        "@type": "sc:Manifest",
        "label": manifest_label,
        "sequences": [
            {
                "@id": sequence_id,
                "@type": "sc:Sequence",
                "label": "Current Page Order",
                "viewingDirection": viewing_direction,
                "canvases": canvases
            }
        ]
    }

    if manifest_description:
        manifest["description"] = manifest_description
    if license_url:
        manifest["license"] = license_url
    if attribution_text:
        manifest["attribution"] = attribution_text

    return manifest

# ==========================================
# 3. 実行処理
# ==========================================

def main():
    # 入力CSVチェック
    if not os.path.exists(INPUT_CSV):
        print(f"エラー: {INPUT_CSV} が見つかりません。同じフォルダに置いてください。")
        return

    try:
        # CSV読み込み
        # encoding='utf-8' を 'shift_jis' または 'cp932' に変更
        with open(INPUT_CSV, encoding='shift_jis', newline='') as f:
            reader = csv.DictReader(f)
            rows = list(reader)

        print(f"処理開始... ({len(rows)} 件のデータ)")

        # マニフェスト生成
        manifest_data = create_single_manifest_v2(rows)

        if manifest_data:
            # 出力 (カレントディレクトリ)
            output_path = os.path.join(OUTPUT_DIR, MANIFEST_FILENAME)
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(manifest_data, f, indent=2, ensure_ascii=False)

            print(f"\n{output_path} にマニフェストを出力しました。")
        else:
            print("マニフェストの生成に失敗しました。")

    except FileNotFoundError:
        print(f"エラー: {INPUT_CSV} が見つかりません。")
    except Exception as e:
        print(f"予期せぬエラーが発生しました: {e}")

if __name__ == "__main__":
    main()

処理開始... (7 件のデータ)

./Anonymous-manifest.json にマニフェストを出力しました。


Colab にファイルをアップロードするには、以下のいずれかの方法を使用できます。

1.  **ファイルブラウザを使用する:**
    *   Colab の左側にあるフォルダアイコン（ファイルブラウザ）をクリックします。
    *   `↑` アップロードアイコンをクリックし、ローカルコンピュータから `Anonymous-data.csv` ファイルを選択します。

2.  **コードを使用する:**
    *   以下のコードセルを実行し、表示されるファイル選択ダイアログから `Anonymous-data.csv` ファイルを選択してアップロードします。


In [13]:
from google.colab import files

# アップロードするファイル名
file_to_upload = 'Anonymous-data.csv'

# ファイルアップロードダイアログを表示
uploaded = files.upload()

# アップロードされたファイルがあるか確認
if file_to_upload in uploaded:
    print(f"'{file_to_upload}' が正常にアップロードされました。")
else:
    print(f"'{file_to_upload}' はアップロードされませんでした。")

Saving Anonymous-data.csv to Anonymous-data (1).csv
'Anonymous-data.csv' はアップロードされませんでした。


ファイルがアップロードされたことを確認するために、現在のディレクトリの内容を表示します。

In [15]:
!ls -F

'Anonymous-data (1).csv'   Anonymous-data.csv   sample_data/


Colab に画像ファイルをアップロードするには、以下のいずれかの方法を使用できます。

1.  **ファイルブラウザを使用する:**
    *   Colab の左側にあるフォルダアイコン（ファイルブラウザ）をクリックします。
    *   `↑` アップロードアイコンをクリックし、ローカルコンピュータからすべての画像ファイルを選択してアップロードします。

2.  **コードを使用する:**
    *   以下のコードセルを実行し、表示されるファイル選択ダイアログからすべての画像ファイルを選択してアップロードします。


In [18]:
from google.colab import files

print("画像ファイルをアップロードしてください（例: e_ground.jpg, e_east1.jpg など）")

# ファイルアップロードダイアログを表示
uploaded_images = files.upload()

for filename in uploaded_images.keys():
    print(f"'{filename}' が正常にアップロードされました。")


画像ファイルをアップロードしてください（例: e_ground.jpg, e_east1.jpg など）


Saving e_east1.jpg to e_east1.jpg
Saving e_east2.jpg to e_east2.jpg
Saving e_east3.jpg to e_east3.jpg
Saving e_ground.jpg to e_ground.jpg
Saving e_north.jpg to e_north.jpg
Saving e_south.jpg to e_south.jpg
Saving e_west.jpg to e_west.jpg
'e_east1.jpg' が正常にアップロードされました。
'e_east2.jpg' が正常にアップロードされました。
'e_east3.jpg' が正常にアップロードされました。
'e_ground.jpg' が正常にアップロードされました。
'e_north.jpg' が正常にアップロードされました。
'e_south.jpg' が正常にアップロードされました。
'e_west.jpg' が正常にアップロードされました。


In [21]:
import json

manifest_filename = "Anonymous-manifest.json"

try:
    with open(manifest_filename, 'r', encoding='utf-8') as f:
        manifest_content = json.load(f)
        print(json.dumps(manifest_content, indent=2, ensure_ascii=False))
except FileNotFoundError:
    print(f"エラー: {manifest_filename} が見つかりません。")
except Exception as e:
    print(f"予期せぬエラーが発生しました: {e}")

{
  "@context": "http://iiif.io/api/presentation/2/context.json",
  "@id": "https://www.u.tsukuba.ac.jp/~s2211717/Anonymous-manifest.json",
  "@type": "sc:Manifest",
  "label": "Anonymous Church Mural Collection",
  "sequences": [
    {
      "@id": "https://www.u.tsukuba.ac.jp/~s2211717/sequence/s1",
      "@type": "sc:Sequence",
      "label": "Current Page Order",
      "viewingDirection": "left-to-right",
      "canvases": [
        {
          "@id": "https://www.u.tsukuba.ac.jp/~s2211717/canvas/e_ground",
          "@type": "sc:Canvas",
          "label": "ground",
          "height": 6000,
          "width": 12000,
          "metadata": [
            {
              "label": "Title",
              "value": "平面図"
            },
            {
              "label": "Description",
              "value": "教会を上から見た図"
            },
            {
              "label": "Hierarchy_Level",
              "value": "Level1=wall"
            },
            {
              "label": "Includes

画像ファイルがアップロードされたことを確認するために、現在のディレクトリの内容を表示します。

In [19]:
!ls -F

'Anonymous-data (1).csv'   e_east2.jpg	  e_north.jpg   sample_data/
 Anonymous-data.csv	   e_east3.jpg	  e_south.jpg
 e_east1.jpg		   e_ground.jpg   e_west.jpg


Colab からファイルをダウンロードするには、以下のいずれかの方法を使用できます。

1.  **ファイルブラウザを使用する:**
    *   Colab の左側にあるフォルダアイコン（ファイルブラウザ）をクリックします。
    *   ダウンロードしたいファイル（例: `Anonymous-manifest.json`, `e_ground.jpg` など）にカーソルを合わせ、縦3点リーダーをクリックします。
    *   「ダウンロード」を選択します。

2.  **コードを使用する:**
    *   以下のコードセルを実行し、`files.download()` 関数を使用して特定のファイルをダウンロードします。


In [22]:
from google.colab import files

# ダウンロードしたいファイル名を指定
files_to_download = [
    'Anonymous-manifest.json',
    'e_ground.jpg',
    'e_east1.jpg',
    'e_east2.jpg',
    'e_east3.jpg',
    'e_north.jpg',
    'e_south.jpg',
    'e_west.jpg'
]

print("ファイルをダウンロードしています...")
for filename in files_to_download:
    if os.path.exists(filename):
        files.download(filename)
        print(f"'{filename}' をダウンロードしました。")
    else:
        print(f"警告: '{filename}' が見つかりませんでした。スキップします。")

print("ダウンロードプロセスが完了しました。")

ファイルをダウンロードしています...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'Anonymous-manifest.json' をダウンロードしました。


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

'e_south.jpg' をダウンロードしました。


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'e_west.jpg' をダウンロードしました。
ダウンロードプロセスが完了しました。
